# WGS pipeline

Orchestrates the genotype-side pipeline end to end: data acquisition, per-callset filter/QC,
normalization, merge, relatedness, dedup, ancestry QC/PCA, and GWAS. Companion to
`clinical_core.ipynb`, which owns the clinical side (`individual_core.csv`,
`genome_crosswalk.csv`, `analysis_grain.csv`, per-callset sex-update files) that this notebook
reads at the points the pipeline needs them.

Runs locally; cells that touch the cluster do so over `ssh`/`sbatch` — you run every cell that
submits a job, same as today's manual commands, just typed once instead of copied from a README.

## 0. Data acquisition — provenance

Where each source callset came from. Kept here (not just run once and discarded) so the pull
is part of the record, and re-runnable if a callset ever needs re-fetching — `synapse get`
resumes rather than restarting.

In [ ]:
import subprocess

# The ONE cluster path in this notebook. Everything remote is derived from it.
# It is the project root on biowulf: code and data share a root there, so this is
# simultaneously $WGS_ROOT, $PROJECT_ROOT and the old $BUNDLE — they are one thing now.
REMOTE_ROOT = "/data/CARDPB2/sysbio/wgs"

HELIX = "helix.nih.gov"   # transfer node — Synapse/large downloads go here, never through biowulf


def run_on_helix(cmd):
    """Run a shell command on helix.nih.gov over ssh, streaming output, raising on failure."""
    subprocess.run(["ssh", HELIX, cmd], check=True)

### DivCo_HS

In [ ]:
DIVCO_DEST = f"{REMOTE_ROOT}/data/amp-ad-genomics/DivCo_HS/joint_calls"
DIVCO_MERGED = "syn68260951"        # merged.deduped.vcf.gz
DIVCO_MERGED_TBI = "syn68260952"    # merged.deduped.vcf.gz.tbi

run_on_helix(f"mkdir -p {DIVCO_DEST} && "
             f"synapse get {DIVCO_MERGED} --downloadLocation {DIVCO_DEST} && "
             f"synapse get {DIVCO_MERGED_TBI} --downloadLocation {DIVCO_DEST}")

### WGS_Harmonization

In [ ]:
WGS_HARM_DEST = f"{REMOTE_ROOT}/data/amp-ad-genomics/WGS_Harmonization/joint_calls"
WGS_HARM_FOLDER = "syn11707420"    # 24 chromosome VCFs (1-22, X, Y)

run_on_helix(f"mkdir -p {WGS_HARM_DEST} && "
             f"synapse get -r {WGS_HARM_FOLDER} --downloadLocation {WGS_HARM_DEST}")

### WB-DWGS

In [ ]:
WB_DWGS_DEST = f"{REMOTE_ROOT}/data/amp-pd-genomics/WB-DWGS/joint_calls"
WB_DWGS_SRC = "gs://amp-pd-genomics/releases/2023_v4release_1027/wgs-WB-DWGS/plink/pfiles/all_chrs_merged.{pgen,psam,pvar,log}"

run_on_helix(f"mkdir -p {WB_DWGS_DEST} && "
             f"gcloud storage cp {WB_DWGS_SRC} {WB_DWGS_DEST}/ --billing-project 8641313829")

### BR-DSNWGS

In [ ]:
BR_DSNWGS_DEST = f"{REMOTE_ROOT}/data/amp-pd-genomics/BR-DSNWGS/joint_calls"
BR_DSNWGS_SRC = "gs://amp-pd-receipt-2026/mssm/20260626-transfer-mssm-jvcf/*"

run_on_helix(f"mkdir -p {BR_DSNWGS_DEST} && "
             f'gcloud storage cp "{BR_DSNWGS_SRC}" {BR_DSNWGS_DEST}/ --billing-project 8641313829')

## 1. VCF → pgen

Only BR-DSNWGS needs this. How each callset got to pgen:

| Callset | Route |
|---|---|
| `wgs_harm` | 24 per-chromosome **b37** VCFs → per-chrom `plink2` filter → `--pmerge-list` → `liftOver` b37→hg38 on the *merged* file → `--chr 1-22,X,Y` |
| `divco_hs` | one pre-merged hg38 VCF → `bcftools` filter → `plink2 --make-pgen` |
| `wb_dwgs` | none — AMP-PD ships plink2 pfiles in hg38 |
| `br_dsnwgs` | this section — same two stages as `divco_hs` |

**chrX needs a sex file.** plink2 refuses to import non-PAR X without per-sample sex, so
stage 2 writes an all-unknown placeholder purely to let the import proceed. Real sex comes
from `clinical_core` §10 and is applied by genetics step 1, where the genotype sex-check
also adjudicates it.

Chromosome naming and variant IDs need **not** match the other callsets —
`02_normalize.sh` re-runs `--output-chr chrM --set-all-var-ids '@:#:$r:$a'` across all of
them. This section only has to produce a valid pgen.

In [ ]:
BIOWULF = "biowulf.nih.gov"   # compute; bulk transfers go to helix

WGS_ROOT = REMOTE_ROOT
BUNDLE = REMOTE_ROOT   # code lives AT the root now, not in a sub-bundle
LOGS = f"{BUNDLE}/logs"


def submit_step(script, name, sbatch_args=(), **exports):
    """Submit a bundle step through submit.sh, which resolves $BUNDLE and sources config.sh.

    sbatch_args overrides the script's own #SBATCH directives (sbatch takes the command line
    over the file). Pass resources here rather than typing them at a shell, so the notebook
    stays a faithful record of what was actually submitted — submit.sh logs them either way,
    but only this way can the run be reproduced from the notebook.
    """
    ex = ",".join(f"{k}={v}" for k, v in exports.items())
    extra = (" " + " ".join(sbatch_args)) if sbatch_args else ""
    r = subprocess.run(["ssh", BIOWULF, f"cd {BUNDLE} && ./submit.sh {script} "
                                        f"--job-name={name}{extra} --export={ex} --parsable"],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr.strip() or r.stdout.strip())
    jid = next((t for t in reversed(r.stdout.split()) if t.isdigit()), "?")
    print(f"job {jid}   tail -f {LOGS}/{name}.o")
    return jid


def sbatch_on_biowulf(name, body, time="12:00:00", cpus=16, mem="64g"):
    """Write <name>.sh on biowulf from `body`, submit it, return the job id.

    For one-off work that has no script in the bundle — currently only §1."""
    script = f"{BUNDLE}/generated/{name}.sh"
    header = (f"#!/bin/bash\n#SBATCH --job-name={name}\n"
              f"#SBATCH --output={LOGS}/{name}.o\n#SBATCH --error={LOGS}/{name}.e\n"
              f"#SBATCH --time={time}\n#SBATCH --cpus-per-task={cpus}\n"
              f"#SBATCH --mem={mem}\n#SBATCH --partition=norm\nset -euo pipefail\n\n")
    subprocess.run(["ssh", BIOWULF, f"mkdir -p $(dirname {script}) {LOGS} && cat > {script}"],
                   input=header + body, text=True, check=True)
    jid = subprocess.run(["ssh", BIOWULF, f"sbatch --parsable {script}"],
                         capture_output=True, text=True, check=True).stdout.strip()
    print(f"job {jid}   tail -f {LOGS}/{name}.o")
    return jid

In [ ]:
BR = f"{WGS_ROOT}/data/amp-pd-genomics/BR-DSNWGS"
BR_VCF = f"{BR}/joint_calls/AMPPD_postmortem_joint_gt_call_97donors.vcf.gz"
BR_FILTERED = f"{BR}/pgen/intermediate/br_dsnwgs_filtered.vcf.gz"
BR_PGEN = f"{BR}/pgen/br_dsnwgs_hg38"

BUILD = "hg38"   # b37 would need a liftover stage instead — see wgs_harm_stage3_liftover.sh

### Stage 1 — filter

Biallelic SNPs, IDs set to `CHROM:POS:REF:ALT`. `--apply-filters 'PASS,.'` keeps passing
and unfiltered sites and drops explicit VQSR failures, so it is correct whether or not this
callset was recalibrated. The job echoes the depositor README and the `##reference` header
first, so the build it actually ran on is on the record.

In [ ]:
s1 = sbatch_on_biowulf("br_dsnwgs_s1_filter", f"""
module load bcftools
mkdir -p $(dirname {BR_FILTERED})

cat {BR}/joint_calls/README.txt || true
bcftools view -h {BR_VCF} | grep -i '^##reference' || echo "no ##reference header"
echo "samples in: $(bcftools query -l {BR_VCF} | wc -l)"

bcftools view \\
    --apply-filters 'PASS,.' \\
    --min-alleles 2 --max-alleles 2 --type snps \\
    {BR_VCF} \\
  | bcftools annotate --set-id '%CHROM:%POS:%REF:%ALT' \\
  | bgzip -@ 8 > {BR_FILTERED}

tabix -p vcf {BR_FILTERED}
echo "variants out: $(bcftools index -n {BR_FILTERED})"
""")

### Stage 2 — VCF → pgen

`--chr 1-22,X,Y` drops alt and unplaced contigs and accepts either `chr1` or `1`.
`--split-par` separates the chrX pseudo-autosomal regions.

In [ ]:
s2 = sbatch_on_biowulf("br_dsnwgs_s2_vcf_to_pgen", f"""
module load bcftools plink/6-alpha
mkdir -p {BR}/pgen

SEX=$(dirname {BR_FILTERED})/placeholder_sex.txt
bcftools query -l {BR_FILTERED} | awk '{{print $1, $1, 0}}' > "$SEX"

plink2 \\
    --vcf {BR_FILTERED} \\
    --chr 1-22,X,Y \\
    --split-par {BUILD} \\
    --update-sex "$SEX" \\
    --make-pgen \\
    --threads 16 \\
    --out {BR_PGEN}

echo "variants: $(grep -vc '^#' {BR_PGEN}.pvar)"
echo "samples:  $(grep -vc '^#' {BR_PGEN}.psam)"
""", time="24:00:00", mem="128g")

### Pull the psam — *no longer needed*

`clinical_core.py` §7 now reads each callset's **raw** psam in place
(`data/<callset>/pgen/*.psam`, `joint_calls/all_chrs_merged.psam`). There is no local copy to
keep in sync, and no `_sexupd` psam in the loop — that file is step 1's *output*, so reading it
here made the clinical side depend on the pipeline it feeds.

## 2. GenoTools — per-callset filter, ancestry, QC

`scripts/01_genotools.sh` runs once per callset: applies the sex file, filters to biallelic
PASS SNPs, converts to bed, then projects onto the reference panel for ancestry and runs QC.
It writes the two things later steps read back — the `<ANC>_pass_fail` JSON (step 5 pulls
QC-fail reasons from it) and the predicted ancestry labels (steps 4 and 6).

From here on every step is a script in `scripts/`, submitted through `submit.sh`. The
notebook supplies the run order, the per-step variables, and the pulls in between; the bash
stays in the bundle so the notebook and the cluster can't drift apart.

Only `br_dsnwgs` needs a run — the other three came through the validation pass.

### Push the scripts — *replaced by git*

This cell used `rsync -av --delete` to overwrite the cluster's code from a laptop. With the
project under version control, the cluster pulls instead, which is versioned, one-directional,
and cannot silently clobber work:

```bash
ssh biowulf.nih.gov 'cd /data/CARDPB2/sysbio/wgs && git pull'
```

The `--delete` made this the most destructive cell in the notebook; it is deliberately gone
rather than merely unused.

In [ ]:
DATA = f"{WGS_ROOT}/data"   # same layout as config.sh and clinical_core.py

# callset -> (base dir, pgen prefix relative to it). Mirrors config.sh's DIR_*/RAW_*.
# Note the asymmetry: WB-DWGS keeps its pgen in joint_calls/, the rest in pgen/.
CALLSETS = {
    "wgs_harm":  (f"{DATA}/amp-ad-genomics/WGS_Harmonization", "pgen/wgs_harm_hg38"),
    "divco_hs":  (f"{DATA}/amp-ad-genomics/DivCo_HS",          "pgen/divco_hs_hg38"),
    "wb_dwgs":   (f"{DATA}/amp-pd-genomics/WB-DWGS",           "joint_calls/all_chrs_merged"),
    "br_dsnwgs": (f"{DATA}/amp-pd-genomics/BR-DSNWGS",         "pgen/br_dsnwgs_hg38"),
}


def genotools(name, cpus=16, mem="128g", time="12:00:00"):
    """Step 1 for one callset.

    cpus is deliberately modest. The Aug-7 br_dsnwgs run died in step 4 with 192 threads on a
    64-CPU allocation: plink2 honours --threads, but genotools' Python stack (OpenBLAS,
    sklearn, numba, xgboost) sizes itself off the NODE core count. 01_genotools.sh now caps
    every pool at $SLURM_CPUS_PER_TASK, so cpus here sets the real thread budget — and since
    each thread carries its own workspace, it sets the memory ceiling too. More is not better.
    """
    base, pgen = CALLSETS[name]
    return submit_step("scripts/01_genotools.sh", f"genotools_{name}",
                       sbatch_args=(f"--cpus-per-task={cpus}", f"--mem={mem}", f"--time={time}"),
                       PGEN=f"{base}/{pgen}",
                       SEX_FILE=f"{WGS_ROOT}/clinical_core_out/{name}_update_sex.txt",
                       OUT_DIR=f"{base}/genotools",
                       DATASET=name)

### Push the sex files — *no longer needed*

Step 1 reads them straight out of `clinical_core_out/` (`config.sh: sex_file`), where §10 wrote
them. Nothing is copied into each callset's `metadata/`, so there is only ever one copy of a
given sex file and it cannot go stale against the run that produced it.

In [ ]:
genotools("br_dsnwgs")